<a href="https://colab.research.google.com/github/adriaanjvv/rule1dash/blob/main/company_analysis_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import gspread
import pandas as pd
from google.auth import default
from google.colab import auth
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)


## Select A google sheet

In [7]:
from googleapiclient.discovery import build

# Create a Google Drive API service client
drive_service = build('drive', 'v3', credentials=creds)

# 3. Query the Drive API to find the 'investment' folder
results = drive_service.files().list(
    q="name='investment' and mimeType='application/vnd.google-apps.folder'",
    spaces='drive',
    fields='files(id, name)').execute()

investment_folders = results.get('files', [])

investment_folder_id = None
if investment_folders:
    investment_folder_id = investment_folders[0]['id']
    print(f"Found 'investment' folder: {investment_folders[0]['name']} (ID: {investment_folder_id})")
else:
    print("Error: 'investment' folder not found.")

# 4. Using the ID of the 'investment' folder, find the 'companies' subfolder
companies_folder_id = None
if investment_folder_id:
    results = drive_service.files().list(
        q=f"name='companies' and mimeType='application/vnd.google-apps.folder' and '{investment_folder_id}' in parents",
        spaces='drive',
        fields='files(id, name)').execute()

    companies_folders = results.get('files', [])

    if companies_folders:
        companies_folder_id = companies_folders[0]['id']
        print(f"Found 'companies' subfolder: {companies_folders[0]['name']} (ID: {companies_folder_id})")
    else:
        print("Error: 'companies' subfolder not found within 'investment' folder.")
else:
    print("Cannot search for 'companies' subfolder without 'investment' folder ID.")

Found 'investment' folder: investment (ID: 1J4agOx0397BHwC9RKgV9Ag-R4c1wQcvs)
Found 'companies' subfolder: companies (ID: 1Z4kSyiN_BCu3yyTsLpj3ly7381cLyIcv)


In [8]:
if companies_folder_id:
    # List all sub-folders within the 'companies' folder
    results = drive_service.files().list(
        q=f"mimeType='application/vnd.google-apps.folder' and '{companies_folder_id}' in parents",
        spaces='drive',
        fields='files(id, name)').execute()

    company_folders = results.get('files', [])

    if company_folders:
        print(f"Found {len(company_folders)} company sub-folders:")
        for folder_info in company_folders:
            print(f"- {folder_info['name']} (ID: {folder_info['id']})")

        # Create a list of tuples for the dropdown options (folder name, folder ID)
        company_folder_options = [(folder['name'], folder['id']) for folder in company_folders]

        # Instantiate the Dropdown widget
        company_folder_dropdown = widgets.Dropdown(
            options=company_folder_options,
            value=company_folder_options[0][1] if company_folder_options else None, # Set default value to the first folder's ID
            description='Select Company Folder:',
            disabled=False,
        )

        # # Display the dropdown widget
        # display(company_folder_dropdown)
    else:
        print("No company sub-folders found within the 'companies' folder.")
else:
    print("Cannot create company folder dropdown without 'companies' folder ID.")

Found 17 company sub-folders:
- lilly (ID: 1YICvTPKeBBuhRA4Xk7CLEs7NejIwoSAw)
- tapestry (ID: 1JyioM6BeuCwEks2Nu7-xPzvbStSlnQ51)
- salesforce (ID: 1bETtFIV9d5oq4616HYgfiqERSBehWkmN)
- aem (ID: 1yDSpTX3KabaRXGphcFyAmYQlc53Uh2rz)
- amzn (ID: 16lY42olG6dnXhLpwfAMFkmsFwDZK6HT3)
- chipotle (ID: 1IdfvWqJ7m56dSh3-_Vlrim5fawFVRyGV)
- dassault (ID: 1l-OHjBvNSAUCuOj6TvCQKtSafnlFqaUg)
- paypal (ID: 1QBOXBnnAPBqxkxrd8hU1Kj8XXnQgfgUR)
- nvda (ID: 1FP5k7cXI6AEy2lj0IXW5fZ5PP9KcjmNk)
- lulu (ID: 1AuXXWIKQJOpAzKWdpBFAuBZ0ZHDSK9pK)
- pfizer (ID: 1mJeuwenrVEPYpqSvZvt2ZfzprDxfQZ30)
- cloudflare (ID: 1TjascAksZqMTCv25ycuN9Nyl9AiVHfRb)
- novo (ID: 1uD76xIVOzGgnS_WdtiboY0N5Hb2ZjO5q)
- uber (ID: 1lGBpT17oFgD1AIvBYLkrQuoqPP1Cv0-Z)
- nxt (ID: 1eMErLySn6TXPIROgyzlz6OEQVwxNKOvJ)
- nike (ID: 1UWnM3LspEfN9KHC05izDdUv5DeBeBxkR)
- decker (ID: 1-SvddKASQvGOZ6N1h-VmSP0oWWEJPdN2)


In [9]:
# Display the dropdown widget
display(company_folder_dropdown)

Dropdown(description='Select Company Folder:', options=(('lilly', '1YICvTPKeBBuhRA4Xk7CLEs7NejIwoSAw'), ('tape…

In [10]:
spreadsheet_dropdown = widgets.Dropdown(
    options=[], # Initially empty
    description='Select Spreadsheet:',
    disabled=False,
)

In [11]:
def list_spreadsheets_in_folder(folder_id):
    """
    Lists all Google Spreadsheets within a given Google Drive folder.

    Args:
        folder_id (str): The ID of the Google Drive folder.

    Returns:
        list: A list of tuples, where each tuple contains the spreadsheet's name and its ID.
              Returns an empty list if no spreadsheets are found or if the folder_id is invalid.
    """
    if not folder_id:
        print("Error: Folder ID cannot be empty.")
        return []

    query = f"mimeType='application/vnd.google-apps.spreadsheet' and '{folder_id}' in parents"
    try:
        results = drive_service.files().list(
            q=query,
            spaces='drive',
            fields='files(id, name)'
        ).execute()

        spreadsheets = results.get('files', [])
        spreadsheet_options = [(spreadsheet['name'], spreadsheet['id']) for spreadsheet in spreadsheets]
        return spreadsheet_options
    except Exception as e:
        print(f"Error listing spreadsheets in folder {folder_id}: {e}")
        return []


In [12]:
def on_company_folder_change(change):
    if change.new:
        selected_folder_id = change.new
        # List spreadsheets in the newly selected folder
        folder_spreadsheets = list_spreadsheets_in_folder(selected_folder_id)

        if folder_spreadsheets:
            # Update the options of the spreadsheet_dropdown
            spreadsheet_dropdown.options = folder_spreadsheets
            # Set the value to the first spreadsheet in the new list, triggering on_spreadsheet_change
            spreadsheet_dropdown.value = folder_spreadsheets[0][1]
            print(f"Spreadsheets loaded for folder ID: {selected_folder_id}")
        else:
            spreadsheet_dropdown.options = []
            spreadsheet_dropdown.value = None # Clear selection if no spreadsheets found
            print(f"No spreadsheets found in folder ID: {selected_folder_id}.")

# Attach the observer to the company folder dropdown
if 'company_folder_dropdown' in locals() and company_folder_dropdown.options:
    company_folder_dropdown.observe(on_company_folder_change, names='value')

    # Trigger initial load for the spreadsheet dropdown based on the initial company folder selection
    if company_folder_dropdown.value:
        initial_folder_spreadsheets = list_spreadsheets_in_folder(company_folder_dropdown.value)
        if initial_folder_spreadsheets:
            spreadsheet_dropdown.options = initial_folder_spreadsheets
            spreadsheet_dropdown.value = initial_folder_spreadsheets[0][1] # Set default spreadsheet
            print(f"Initial spreadsheets loaded for folder ID: {company_folder_dropdown.value}")
        else:
            print(f"No spreadsheets found in the initially selected folder ID: {company_folder_dropdown.value}.")
else:
    print("Company folder dropdown not initialized or has no options.")

Initial spreadsheets loaded for folder ID: 1AuXXWIKQJOpAzKWdpBFAuBZ0ZHDSK9pK


In [13]:


display(spreadsheet_dropdown)

Dropdown(description='Select Spreadsheet:', options=(('lulu-202604', '11ZzVOBYo5RJV-S-agVzSs8XgsAQaANafjcxbcUE…

## Load the spreadsheet

In [31]:
def set_index_and_cols(df):
  # print(f"DEBUG (set_index_and_cols): Initial DataFrame received (head):\n{df.head()}")
  # print(f"DEBUG (set_index_and_cols): Initial DataFrame columns: {list(df.columns)}")
  # print(f"DEBUG (set_index_and_cols): Initial DataFrame shape: {df.shape}")

  # Assume the first row contains dates (after the first cell) and the first column contains metric names
  # Create DataFrame with first row as columns, and then drop that row
  if df.empty or df.shape[0] < 1:
      raise ValueError("Input DataFrame is empty or has no data rows.")

  new_header = df.iloc[0] # Grab the first row for the headers
  df = df[1:].copy()      # Take the data part below the header row
  df.columns = new_header # Set the header row as the new column names

  # Debug print 1: After initial column assignment and row drop
  # print(f"DEBUG (set_index_and_cols): After initial column assign/row drop - Columns: {list(df.columns)}")
  # print(f"DEBUG (set_index_and_cols): After initial column assign/row drop - Head:\n{df.head()}")

  # Check if the DataFrame became empty after dropping the header row (e.g., only one row in original data)
  if df.empty:
      raise ValueError("DataFrame became empty after removing header row.")

  # Rename the first column (which contains metric names) to a descriptive name, e.g., 'Metric'
  # Make sure the column exists before renaming
  if df.columns.empty:
      raise ValueError("DataFrame has no columns after header processing.")

  original_first_column_name = df.columns[0]
  if original_first_column_name == 'Metric': # Avoid renaming if already named 'Metric'
      pass
  else:
      df = df.rename(columns={original_first_column_name: 'Metric'})

  # Debug print 2: After renaming the first column
  # print(f"DEBUG (set_index_and_cols): After renaming first col to 'Metric' - Columns: {list(df.columns)}")
  # print(f"DEBUG (set_index_and_cols): After renaming first col to 'Metric' - Head:\n{df.head()}")

  # Set 'Metric' column as the index
  if 'Metric' not in df.columns:
      raise ValueError(f"Column 'Metric' not found for setting index. Available columns: {list(df.columns)}")
  df = df.set_index('Metric')

  # Transpose the DataFrame so dates are the index and metrics are columns
  df = df.T

  # The index now contains the dates and 'TTM' if present
  df.index.name = 'Date' # Set index name to Date for consistency

  # Handle 'TTM' in the index
  if 'TTM' in df.index:
      df = df.drop(index='TTM')
      print("INFO (set_index_and_cols): 'TTM' row found in index and dropped.")

  # print(f"DEBUG (set_index_and_cols): Index before datetime conversion: {df.index.tolist()}")
  # print(f"DEBUG (set_index_and_cols): Index type before datetime conversion: {type(df.index)}")

  try:
    df.index = pd.to_datetime(df.index, errors='coerce')
    # Filter out rows where the index is NaT (Not a Time) after conversion
    df = df[df.index.notna()]
  except Exception as e_date_processing:
    raise RuntimeError(f"Error during date index processing: {e_date_processing}. Index values: {df.index.tolist() if df is not None else 'df is None'}") from e_date_processing

  # print(f"DEBUG (set_index_and_cols): Final DataFrame shape: {df.shape}")
  # print(f"DEBUG (set_index_and_cols): Final DataFrame columns: {list(df.columns)}")
  # print(f"DEBUG (set_index_and_cols): Final DataFrame head:\n{df.head()}")

  return df

In [17]:
worksheets_list = ['Income-Annual','Balance-Sheet-Annual','Cash-Flow-Annual','Ratios-Annual','Income-TTM','Balance-Sheet-TTM','Cash-Flow-TTM','Ratios-TTM']

In [32]:
import traceback

def load_sheets_into_dict(spreadsheet_obj):
    global sheets_dict # Declare sheets_dict as global to modify it
    global sheets_title
    loaded_sheets = {}
    for sheet_name in worksheets_list:
        try:
            worksheet_data = spreadsheet_obj.worksheet(sheet_name).get()
            try:
                processed_df = set_index_and_cols(pd.DataFrame(worksheet_data))
                loaded_sheets[sheet_name] = processed_df
            except ValueError as ve:
                print(f"ERROR (set_index_and_cols): Failed to process '{sheet_name}' from '{spreadsheet_obj.title}': {ve}. Skipping.")
            except Exception as e_internal:
                print(f"ERROR (set_index_and_cols): An unexpected error occurred while processing '{sheet_name}' from '{spreadsheet_obj.title}': {type(e_internal).__name__}: {e_internal}. Skipping.")
                # traceback.print_exc()
        except gspread.exceptions.WorksheetNotFound:
            print(f"Warning: Worksheet '{sheet_name}' not found in spreadsheet '{spreadsheet_obj.title}'. Skipping.")
        except Exception as e:
            print(f"Error loading worksheet '{sheet_name}' from '{spreadsheet_obj.title}': {type(e).__name__}: {e}. Skipping.")
            # traceback.print_exc()

    sheets_dict = loaded_sheets
    sheets_title = spreadsheet_obj.title
    print(f"Successfully loaded the following worksheets from '{spreadsheet_obj.title}' into sheets_dict: {list(sheets_dict.keys())}")

def on_spreadsheet_change(change):
    if change['new']: # Ensure a value is selected
        selected_spreadsheet_id = change['new']
        selected_spreadsheet = gc.open_by_key(selected_spreadsheet_id)
        print(f"\n--- Worksheets in selected spreadsheet '{selected_spreadsheet.title}' ---")
        try:
            all_worksheets = selected_spreadsheet.worksheets()
            for ws in all_worksheets:
                print(f"- {ws.title}")
        except Exception as e:
            print(f"Error listing worksheets: {e}")
        print("--------------------------------------------------")
        load_sheets_into_dict(selected_spreadsheet)

# Attach the observer to the dropdown
spreadsheet_dropdown.observe(on_spreadsheet_change, names='value')

# Initial load of the selected spreadsheet (which is the default value)
# This ensures sheets_dict is populated when the notebook starts or this cell is run.
# if spreadsheet_dropdown.value:
#     initial_spreadsheet = gc.open_by_key(spreadsheet_dropdown.value)
#     load_sheets_into_dict(initial_spreadsheet)

In [19]:
# Explicitly trigger the on_spreadsheet_change for the current value of spreadsheet_dropdown
if spreadsheet_dropdown.value:
    on_spreadsheet_change({'new': spreadsheet_dropdown.value})



--- Worksheets in selected spreadsheet 'lulu-202604' ---
- Sheet2
- Income-Annual
- Balance-Sheet-Annual
- Cash-Flow-Annual
- Ratios-Annual
- Income-Quarterly
- Balance-Sheet-Quarterly
- Cash-Flow-Quarterly
- Ratios-Quarterly
- Income-TTM
- Balance-Sheet-TTM
- Cash-Flow-TTM
- Ratios-TTM
--------------------------------------------------
DEBUG (set_index_and_cols): Initial DataFrame received (head):
                0      1           2           3           4           5   \
0             Date    TTM  2026-02-01  2025-02-02  2024-01-28  2023-01-29   
1          Revenue  11103       11103       10588        9619        8111   
2   Revenue Growth  4.86%       4.86%      10.07%      18.60%      29.63%   
3  Cost of Revenue   4818        4818        4317        4010        3618   
4     Gross Profit   6284        6284        6271        5609        4492   

           6           7           8           9   ...          14  \
0  2022-01-30  2021-01-31  2020-02-02  2019-02-03  ...  2014-02-

In [20]:
sheets_of_interest = {
    "Ratios-Annual":[
        {"short_name": "ROIC", "orig_name": "Return on Invested Capital (ROIC)"},
         {"short_name": "PE", "orig_name": "PE Ratio"}
    ],
    "Income-Annual":[
        {"short_name": "EPS", "orig_name": "EPS (Diluted)"},
        {"short_name": "Rev", "orig_name": "Revenue"},
        {"short_name": "OpM", "orig_name": "Operating Margin"},
        {"short_name": "OpI", "orig_name": "Operating Income"},
        {"short_name": "SharesO", "orig_name": "Shares Outstanding (Diluted)"},
        {"short_name": "NetI", "orig_name": "Net Income"},
        {"short_name": "DepAm", "orig_name": "Depreciation & Amortization Expenses"}, # Corrected name
        {"short_name": "IncTax", "orig_name": "Provision for Income Taxes"}, # Corrected name
        {"short_name": "EBIT_Margin", "orig_name": "EBIT Margin"}
    ],
    "Balance-Sheet-Annual":[
        {"short_name": "ShareEquity", "orig_name": "Shareholders Equity"},
        {"short_name": "LTDebt", "orig_name": "Total Debt"}, # Corrected name, assuming Total Debt is the closest match for Long-Term Debt
        {"short_name": "AccPay", "orig_name": "Accounts Payable"},
        {"short_name": "Recs", "orig_name": "Accounts Receivable"} # Corrected name
    ],
    "Cash-Flow-Annual":[
        {"short_name": "FCF", "orig_name": "Free Cash Flow"},
        {"short_name": "OCF", "orig_name": "Operating Cash Flow"},
        {"short_name": "Capex", "orig_name": "Capital Expenditures"}
    ]
}

In [21]:
new_rows_structure = {}
for sheet_name, metrics_list in sheets_of_interest.items():
    for metric_dict in metrics_list:
        short_name = metric_dict["short_name"]
        orig_name = metric_dict["orig_name"]
        new_rows_structure[short_name] = {"sheet": sheet_name, "orig_name": orig_name}
rows = new_rows_structure
# display(rows)

In [22]:
new_rows_structure = {}
for sheet_name, metrics_list in sheets_of_interest.items():
    for metric_dict in metrics_list:
        short_name = metric_dict["short_name"]
        orig_name = metric_dict["orig_name"]
        new_rows_structure[short_name] = {"sheet": sheet_name.replace("Annual","TTM"), "orig_name": orig_name}
rows_ttm = new_rows_structure
# display(rows_ttm)

In [23]:
def compile_financial_data(sheets_dict, rows):
    compiled_df_data = []

    for short_name, metric_info in rows.items():
        sheet = metric_info["sheet"]
        orig_name = metric_info["orig_name"]
        if sheet in sheets_dict and orig_name in sheets_dict[sheet].columns:
            series = sheets_dict[sheet][orig_name]
            series.name = short_name  # Rename the series to the short_name for the column
            compiled_df_data.append(series)
        else:
            print(f"Warning: Could not find '{orig_name}' in sheet '{sheet}' for short name '{short_name}'. Skipping.")

    # Concatenate all series into a single DataFrame, aligning by index (Date)
    if compiled_df_data:
        final_df = pd.concat(compiled_df_data, axis=1, join='outer')
        # Sort the DataFrame by its index (Date)
        final_df = final_df.sort_index()
        return final_df
    else:
        print("No data found to compile into a DataFrame.")
        return pd.DataFrame()



In [24]:
def convert_df_to_numeric(df, percentage_cols):
    """
    Converts DataFrame columns to numeric types, handling percentage strings.

    Args:
        df (pd.DataFrame): The input DataFrame.
        percentage_cols (list): A list of column names that contain percentage strings
                                (e.g., '120%') that should be converted to decimals.

    Returns:
        pd.DataFrame: The DataFrame with converted numeric types.
    """
    df_numeric = df.copy()

    for col in df.columns:
        if col in percentage_cols:
            # Convert percentage strings to float decimals (e.g., '120%' -> 1.2)
            df_numeric[col] = df_numeric[col].astype(str).str.replace('%', '', regex=False)
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce') / 100
        else:
            # Convert other columns to numeric, coercing errors
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')
    return df_numeric

In [25]:
def calc_cagr(numeric_values):
  """
  Calculate CAGR for each historical year in the series relative to the latest value.
  The CAGR for the latest date itself will be NaN.
  """
  if numeric_values.empty or len(numeric_values) < 2:
      # Return an empty Series with appropriate type and index
      return pd.Series(dtype=float, index=numeric_values.index, name="CAGR")

  latest_value = numeric_values.iloc[-1]
  latest_date = numeric_values.index[-1]

  cagr_results = {}

  # Iterate through all values except the latest one as historical_value
  for i in range(len(numeric_values) - 1):
      date = numeric_values.index[i]
      historical_value = numeric_values.iloc[i]

      # Ensure historical_value is not zero to avoid division by zero
      if historical_value == 0:
          cagr_results[date] = float('nan')
          continue

      # Calculate time_span in years
      number_of_years = (latest_date.year - date.year)
      number_of_months = (latest_date.month - date.month)
      month_decimal = (number_of_months / 12.0)
      time_span = number_of_years + month_decimal

      # Ensure time_span is positive for CAGR calculation
      if time_span <= 0:
          cagr_results[date] = float('nan')
          continue

      ratio = latest_value / historical_value

      # CAGR is generally not well-defined for negative growth ratios over non-integer time periods.
      # Return NaN to avoid RuntimeWarning: invalid value encountered in scalar power.
      if ratio < 0:
          cagr_results[date] = float('nan')
          continue

      try:
          cagr = (ratio ** (1 / time_span)) - 1
          cagr_results[date] = cagr
      except Exception as e:
          # Catch any other potential mathematical errors
          print(f"An error occurred during CAGR calculation for date={date}: {e}")
          cagr_results[date] = float('nan')

  # Create a Series from the results and reindex it to the original numeric_values index.
  # This will automatically place NaNs for dates where CAGR was not calculated (e.g., the latest date).
  result_series = pd.Series(cagr_results, name="CAGR").reindex(numeric_values.index)
  return result_series

In [26]:
import pandas as pd

def calc_nyear_growthrate(numeric_values, nyears):
    """
    Calculate the n-year compound annual growth rate (CAGR) for each point in a time series,
    looking back approximately 'nyears' from the current point.

    For every date in the `values` (which is sorted by date index) pandas series,
    calculate the CAGR over the previous `nyears`.

    Args:
        values (pd.Series): A pandas Series with a DateTimeIndex and numeric values.
                            Should be sorted by index in ascending order.
        nyears (int): The number of years for the growth rate calculation window.

    Returns:
        pd.Series: A Series containing the n-year CAGR for each applicable date.
                   Dates for which an n-year lookback is not possible will have NaN.
    """
    if numeric_values.empty:
        return pd.Series(dtype=float)

    if len(numeric_values) < 2 or nyears <= 0:
        return pd.Series(dtype=float)

    growth_rates = {}

    # Iterate through each point as the 'end_point' of the n-year window
    for i in range(len(numeric_values)):
        end_date = numeric_values.index[i]
        end_value = numeric_values.iloc[i]

        # Calculate the exact target start date (end_date minus nyears)
        target_start_date = end_date - pd.DateOffset(years=nyears-1, months=11)

        # Find the index of the data point that is closest to or before target_start_date
        # using searchsorted. 'right' finds the first element > target_start_date, so -1 for <=.
        start_idx = numeric_values.index.searchsorted(target_start_date, side='right') - 1

        # Check if a valid start_idx was found and it's not the same as the end_idx
        if start_idx < 0 or start_idx >= i:
            growth_rates[end_date] = float('nan')
            continue

        start_date = numeric_values.index[start_idx]
        start_value = numeric_values.iloc[start_idx]

        # Ensure start_value is not zero or NaN to avoid division errors
        if pd.isna(start_value) or start_value == 0:
            growth_rates[end_date] = float('nan')
            continue

        # Calculate the actual time difference in years. This is used to ensure
        # that the found start_date is reasonably 'nyears' prior. A tolerance can be added if needed.
        actual_time_delta_years = (end_date - start_date).days / 365.25

        # Add a check that the actual time delta is sufficiently close to nyears
        # For example, if we ask for 5-year growth, and the closest data point is only 1 year ago,
        # the calculation might be misleading. Using a tolerance of +/- 1 year.
        if not (nyears - 1 <= actual_time_delta_years <= nyears + 1):
            growth_rates[end_date] = float('nan')
            continue

        # Handle cases where values might be negative or transition across zero
        # CAGR formula is typically for positive values, or values of the same sign.
        if (end_value >= 0 and start_value > 0) or (end_value < 0 and start_value < 0):
            try:
                # Use 'nyears' as the exponent for the CAGR calculation as per the request
                cagr = ((end_value / start_value) ** (1 / nyears)) - 1
                growth_rates[end_date] = cagr
            except (OverflowError, ValueError):
                growth_rates[end_date] = float('nan')
        else:
            growth_rates[end_date] = float('nan')

    # Create a Series, ensuring the index matches the original `numeric_values` index.
    # Fill missing dates with NaN for consistency if not all dates could be calculated.
    result_series = pd.Series(growth_rates).reindex(numeric_values.index)
    result_series.name = f"{nyears}_Year_CAGR"
    return result_series

In [27]:
def calculate_growth_rate(values, mode='simple_cagr'):
  """
  Implement a number of ways to calculate a growth rate from a time series.
  Method:
  'simple_cagr': Calculate the CGAR for each year in the time series values. From the latest year calculate the growth rate for each historic year.
  Input: values is a pandas series with a date index.
  """
  # Ensure values are numeric and sorted by index
  numeric_values = pd.to_numeric(values.sort_index(ascending=True), errors='coerce').dropna()


  if mode == 'simple_cagr':
      growth = calc_cagr(numeric_values)

  if mode == '3year':
      growth = calc_nyear_growthrate(numeric_values, 3)

  return growth

In [28]:
def add_growth_columns(df, cols_to_grow, modes=['simple_cagr','3year']):
    """
    Calculates growth rates for specified columns in a DataFrame and adds them as new columns.

    Args:
        df (pd.DataFrame): The input DataFrame.
        cols_to_grow (list): A list of column names in the DataFrame for which to calculate growth rates.
        modes (list, optional): The growth calculation modes. Defaults: ['simple_cagr', '3year'].


    Returns:
        pd.DataFrame: The DataFrame with new growth rate columns added.
    """
    df_with_growth = df.copy()

    for col_name in cols_to_grow:
      for mode in modes:
        if col_name in df.columns:
            # Calculate the growth rate for the current column
            growth_series = calculate_growth_rate(df[col_name], mode=mode)

            # Create a new column name based on the original column and growth series name
            # The growth_series.name is set within calculate_growth_rate (e.g., 'CAGR', '3_Year_CAGR')
            new_col_name = f"{col_name}_{growth_series.name}"

            # Add the new growth rate series as a column to the DataFrame
            # Using .to_frame() and .join() to ensure alignment by index
            df_with_growth = df_with_growth.join(growth_series.to_frame(name=new_col_name))
        else:
            print(f"Warning: Column '{col_name}' not found in the DataFrame. Skipping growth calculation for this column.")

    return df_with_growth

In [29]:
def plot_financial_metrics_and_growth(data_df, sheets_title):
    metrics_to_plot = ['Rev', 'FCF', 'ShareEquity', 'EPS']

    # Define a color palette for consistency across metrics and their growth rates
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'] # Added color for PE
    color_map = {metric: color for metric, color in zip(metrics_to_plot + ['PE'], colors)}

    # Create subplots: one for main metrics, one for their growth rates, one for EPS, and one for PE
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        subplot_titles=(
                            f"**{sheets_title}** - Financial Metrics (Rev, FCF, ShareEquity)",
                            "Growth Rates (Rev, FCF, ShareEquity, EPS)",
                            "EPS",
                            "PE Ratio"
                        ))

    # --- First Subplot (Row 1): Financial Metrics (Rev, FCF, ShareEquity) ---
    for metric_name in ['Rev', 'FCF', 'ShareEquity']:
        fig.add_trace(
            go.Scatter(
                x=data_df.index,
                y=data_df[metric_name],
                mode='lines+markers',
                name=metric_name,
                line=dict(color=color_map[metric_name]),
                yaxis='y' # Assign to the primary y-axis of the first subplot
            ),
            row=1, col=1
        )

    # --- Second Subplot (Row 2): Growth Rates (Rev, FCF, ShareEquity, EPS) ---
    for metric_name in metrics_to_plot:
        # Add simple CAGR
        fig.add_trace(
            go.Scatter(
                x=data_df.index,
                y=data_df[f'{metric_name}_CAGR'],
                mode='lines+markers',
                name=f'{metric_name} CAGR',
                line=dict(color=color_map[metric_name], dash='solid'),
                showlegend=True,
                yaxis='y2'
            ),
            row=2, col=1
        )
        # Add 3-Year CAGR
        fig.add_trace(
            go.Scatter(
                x=data_df.index,
                y=data_df[f'{metric_name}_3_Year_CAGR'],
                mode='lines+markers',
                name=f'{metric_name} 3-Year CAGR',
                line=dict(color=color_map[metric_name], dash='dot'),
                showlegend=True,
                yaxis='y2'
            ),
            row=2, col=1
        )

    # --- Third Subplot (Row 3): EPS ---
    # Add EPS
    fig.add_trace(
        go.Scatter(
            x=data_df.index,
            y=data_df['EPS'],
            mode='lines+markers',
            name='EPS',
            line=dict(color=color_map['EPS']),
            yaxis='y3'
        ),
        row=3, col=1
    )

    # --- Fourth Subplot (Row 4): PE Ratio ---
    fig.add_trace(
        go.Scatter(
            x=data_df.index,
            y=data_df['PE'],
            mode='lines+markers',
            name='PE Ratio',
            line=dict(color=color_map['PE']),
            yaxis='y4'
        ),
        row=4, col=1
    )

    # Update axis properties explicitly using update_yaxes for clarity
    fig.update_yaxes(title_text='Value', row=1, col=1, showgrid=True, visible=True)
    fig.update_yaxes(title_text="Growth Rate", row=2, col=1, showgrid=True, visible=True)
    fig.update_yaxes(title_text='EPS', row=3, col=1, showgrid=True, visible=True)
    fig.update_yaxes(title_text='PE Ratio', row=4, col=1, showgrid=True, visible=True)

    # General layout updates
    fig.update_layout(
        title_text=f"**{sheets_title}** Financial Metrics and Their Growth Rates",
        height=1200, # Increased height to accommodate 4 subplots
        hovermode="x unified",
        legend_title_text="Metrics"
    )

    fig.show()

In [30]:
def compile_financial_data(sheets_dict, rows):
    compiled_df_data = []

    for short_name, metric_info in rows.items():
        sheet = metric_info["sheet"]
        orig_name = metric_info["orig_name"]
        if sheet in sheets_dict and orig_name in sheets_dict[sheet].columns:
            series = sheets_dict[sheet][orig_name]
            series.name = short_name  # Rename the series to the short_name for the column
            compiled_df_data.append(series)
        else:
            print(f"Warning: Could not find '{orig_name}' in sheet '{sheet}' for short name '{short_name}'. Skipping.")

    # Concatenate all series into a single DataFrame, aligning by index (Date)
    if compiled_df_data:
        final_df = pd.concat(compiled_df_data, axis=1, join='outer')
        # Sort the DataFrame by its index (Date)
        final_df = final_df.sort_index()
        return final_df
    else:
        print("No data found to compile into a DataFrame.")
        return pd.DataFrame()

final_df = compile_financial_data(sheets_dict, rows)

print("\n--- Debugging Sheet and Column Names ---")
if not sheets_dict:
    print("No sheets were loaded into sheets_dict. Please check 'worksheets_list' and the selected spreadsheet.")
else:
    print("Sheets loaded successfully:")
    for sheet_name, df in sheets_dict.items():
        print(f"  - '{sheet_name}' (Columns: {list(df.columns)})")

print("\n--- Expected Columns from 'rows' (sheets_of_interest) ---")
for short_name, metric_info in rows.items():
    print(f"  - Short Name: '{short_name}', Expected Sheet: '{metric_info["sheet"]}', Expected Column: '{metric_info["orig_name"]}'")


--- Debugging Sheet and Column Names ---
Sheets loaded successfully:
  - 'Income-Annual' (Columns: ['Revenue', 'Revenue Growth', 'Cost of Revenue', 'Gross Profit', 'Selling, General & Admin', 'Depreciation & Amortization Expenses', 'Other Operating Expenses', 'Operating Income', 'Interest Income', 'Interest Expense', 'Other Non-Operating Income (Expense)', 'Total Non-Operating Income (Expense)', 'Pretax Income', 'Provision for Income Taxes', 'Net Income', 'Minority Interest in Earnings', 'Earnings From Discontinued Operations', 'Net Income Growth', 'Shares Outstanding (Basic)', 'Shares Outstanding (Diluted)', 'Shares Change', 'EPS (Basic)', 'EPS (Diluted)', 'EPS Growth', 'Shares Outstanding', 'Free Cash Flow Growth', 'Free Cash Flow Per Share', 'Gross Margin', 'Operating Margin', 'Profit Margin', 'Free Cash Flow Margin', 'EBITDA', 'EBITDA Margin', 'EBIT', 'EBIT Margin', 'Effective Tax Rate', 'Total Operating Expenses'])
  - 'Balance-Sheet-Annual' (Columns: ['Cash & Equivalents', 'Cash

In [ ]:
# with_growth_df.columns